# IPP Lab Assignment — Problems from the Last 3 Slides
Based on `IPP_Lab9.pptx`, slides 13–15. Includes all six lab problems: edge detection comparison, noise sensitivity, RLE vs Huffman, JPEG-style compression, GLCM texture segmentation, and multi-feature disease segmentation.

**Input images:** `natural.jpg`, `smooth.png`, `city.jpg`, `textures.jpg`, `plant_disease.jpg`. Keep them beside this notebook. `natural.jpg` is used for Problems 1 and 2. Install dependencies if needed: `python -m pip install opencv-python numpy matplotlib scikit-learn scikit-image`.

## Imports and helper functions

In [ ]:
import os, heapq
from collections import Counter
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage.feature import graycomatrix, graycoprops
from skimage.metrics import structural_similarity

def gray_image(path):
    if not os.path.exists(path):
        print(f"Missing input image: {path}. Place it beside this notebook.")
        return None
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None: print(f"Could not read {path}.")
    return im

def color_image(path):
    if not os.path.exists(path):
        print(f"Missing input image: {path}. Place it beside this notebook.")
        return None
    im = cv2.imread(path, cv2.IMREAD_COLOR)
    if im is None: print(f"Could not read {path}.")
    return im

def show_images(images, titles, figsize=(14, 4), cmap="gray"):
    plt.figure(figsize=figsize)
    for i, (im, title) in enumerate(zip(images, titles), 1):
        plt.subplot(1, len(images), i)
        plt.imshow(im, cmap=cmap); plt.title(title); plt.axis("off")
    plt.tight_layout(); plt.show()

print("Imports completed.")

## Problem 1 — Comparative Edge Detection on Natural Images
Apply Sobel, Laplacian, and Canny to an outdoor natural image and compare their outputs. Input: `natural.jpg`.

In [ ]:
im = gray_image("natural.jpg")
if im is not None:
    sx = cv2.Sobel(im, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(im, cv2.CV_64F, 0, 1, ksize=3)
    sobel = cv2.convertScaleAbs(cv2.magnitude(sx, sy))
    lap = cv2.convertScaleAbs(cv2.Laplacian(im, cv2.CV_64F, ksize=3))
    canny = cv2.Canny(im, 100, 200)
    show_images([im, sobel, lap, canny],
                ["Original grayscale", "Sobel magnitude", "Laplacian", "Canny"],
                figsize=(15, 4))
    print("Analysis: Sobel highlights gradients; Laplacian responds to rapid intensity changes and noise; Canny often gives thinner, connected edges through non-maximum suppression and hysteresis thresholds. The most meaningful result depends on image content and threshold settings.")

## Problem 2 — Noise Sensitivity Experiment
Add Gaussian and salt-and-pepper noise to a clean grayscale image, apply Sobel and Laplacian, and assess noise sensitivity and preprocessing. Input: `natural.jpg`.

In [ ]:
clean = gray_image("natural.jpg")
if clean is not None:
    rng = np.random.default_rng(42)
    noisy_gaussian = np.clip(clean.astype(float) + rng.normal(0, 20, clean.shape), 0, 255).astype(np.uint8)
    noisy_sp = clean.copy()
    rnd = rng.random(clean.shape)
    noisy_sp[rnd < 0.02] = 0
    noisy_sp[rnd > 0.98] = 255

    def edge_maps(a):
        x = cv2.Sobel(a, cv2.CV_32F, 1, 0, ksize=3)
        y = cv2.Sobel(a, cv2.CV_32F, 0, 1, ksize=3)
        return (cv2.convertScaleAbs(cv2.magnitude(x, y)),
                cv2.convertScaleAbs(cv2.Laplacian(a, cv2.CV_32F, ksize=3)))

    show_images([clean, noisy_gaussian, noisy_sp],
                ["Clean image", "Gaussian noise", "Salt-and-pepper noise"], figsize=(12, 4))
    s0, l0 = edge_maps(clean); sg, lg = edge_maps(noisy_gaussian); ss, ls = edge_maps(noisy_sp)
    show_images([s0, l0, sg, lg, ss, ls],
                ["Clean Sobel", "Clean Laplacian", "Gaussian Sobel", "Gaussian Laplacian",
                 "Salt-pepper Sobel", "Salt-pepper Laplacian"], figsize=(16, 7))
    g_blur = cv2.GaussianBlur(noisy_gaussian, (5, 5), 0)
    sp_blur = cv2.medianBlur(noisy_sp, 5)
    s1, l1 = edge_maps(g_blur); s2, l2 = edge_maps(sp_blur)
    show_images([s1, l1, s2, l2],
                ["Gaussian blur + Sobel", "Gaussian blur + Laplacian",
                 "Median blur + Sobel", "Median blur + Laplacian"], figsize=(14, 4))
    print("Observations: Laplacian is generally more noise-sensitive because second derivatives amplify rapid changes. Noise adds false edges and reduces clarity. Gaussian blur helps Gaussian noise; median blur often helps salt-and-pepper noise.")

## Problem 3 — Compression Ratio Comparison (RLE vs Huffman)
Load a grayscale image with large smooth regions (`smooth.png`). Report original and compressed bit lengths and percentage reduction. Estimates below count 8 bits per original pixel, 8 bits per RLE value plus 32 bits per run length, and Huffman payload bits only (excluding code-table/header overhead).

In [ ]:
im = gray_image("smooth.png")
if im is not None:
    pix = im.ravel().tolist()
    original_bits = len(pix) * 8
    runs = []
    value, length = pix[0], 1
    for p in pix[1:]:
        if p == value and length < 2**32 - 1: length += 1
        else: runs.append((value, length)); value, length = p, 1
    runs.append((value, length))
    rle_bits = len(runs) * 40

    freq = Counter(pix)
    heap, uid = [], 0
    for symbol, count in freq.items():
        heapq.heappush(heap, (count, uid, {symbol: 0})); uid += 1
    if len(heap) == 1:
        huff_bits = len(pix)
    else:
        while len(heap) > 1:
            f1, _, a = heapq.heappop(heap); f2, _, b = heapq.heappop(heap)
            merged = {s: d + 1 for s, d in a.items()}
            merged.update({s: d + 1 for s, d in b.items()})
            heapq.heappush(heap, (f1 + f2, uid, merged)); uid += 1
        lengths = heap[0][2]
        huff_bits = sum(freq[s] * lengths[s] for s in freq)

    reduction = lambda n: (1 - n / original_bits) * 100
    print("Dimensions:", im.shape)
    print("Original bits:", original_bits)
    print("RLE run count:", len(runs), "| RLE bits estimate:", rle_bits,
          f"| reduction: {reduction(rle_bits):.2f}%")
    print("Huffman payload bits:", huff_bits, f"| reduction: {reduction(huff_bits):.2f}%")
    print("Explanation: RLE benefits from long runs of repeated pixels. Huffman benefits from unequal symbol frequencies. Actual files also need metadata and code-table storage.")

## Problem 4 — JPEG Block Artifact Analysis
Implement 8×8 DCT, JPEG quantisation, inverse DCT, and compare low/medium/high quantisation on `city.jpg`. Report MSE, PSNR, and SSIM.

In [ ]:
bgr = color_image("city.jpg")
if bgr is not None:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ycc = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb).astype(np.float32)
    Q = np.array([[16,11,10,16,24,40,51,61],[12,12,14,19,26,58,60,55],
      [14,13,16,24,40,57,69,56],[14,17,22,29,51,87,80,62],
      [18,22,37,56,68,109,103,77],[24,35,55,64,81,104,113,92],
      [49,64,78,87,103,121,120,101],[72,92,95,98,112,100,103,99]], dtype=np.float32)

    def compress(ycc, strength):
        out = np.empty_like(ycc); q = np.clip(Q * strength, 1, 255)
        h, w = ycc.shape[:2]; ph = (8-h%8)%8; pw = (8-w%8)%8
        for ch in range(3):
            plane = cv2.copyMakeBorder(ycc[:,:,ch], 0, ph, 0, pw, cv2.BORDER_REFLECT)
            restored = np.zeros_like(plane)
            for r in range(0, plane.shape[0], 8):
                for c in range(0, plane.shape[1], 8):
                    block = plane[r:r+8,c:c+8] - 128
                    coeff = cv2.dct(block)
                    restored[r:r+8,c:c+8] = cv2.idct(np.round(coeff/q)*q) + 128
            out[:,:,ch] = restored[:h,:w]
        return np.clip(out, 0, 255).astype(np.uint8)

    strengths = [0.5, 1.0, 2.0]
    names = ["Low quantisation", "Medium quantisation", "High quantisation"]
    restored_rgb = []
    for strength in strengths:
        restored_ycc = compress(ycc, strength)
        restored_rgb.append(cv2.cvtColor(cv2.cvtColor(restored_ycc, cv2.COLOR_YCrCb2BGR), cv2.COLOR_BGR2RGB))

    plt.figure(figsize=(14,4))
    plt.subplot(1,4,1); plt.imshow(rgb); plt.title("Original"); plt.axis("off")
    for i, (arr, name) in enumerate(zip(restored_rgb, names), 2):
        plt.subplot(1,4,i); plt.imshow(arr); plt.title(name); plt.axis("off")
    plt.tight_layout(); plt.show()

    diff = cv2.absdiff(rgb, restored_rgb[-1])
    heat = cv2.applyColorMap(cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY), cv2.COLORMAP_JET)
    plt.figure(figsize=(6,4)); plt.imshow(cv2.cvtColor(heat, cv2.COLOR_BGR2RGB))
    plt.title("Difference heatmap — high quantisation"); plt.axis("off"); plt.show()

    orig_gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    for name, arr in zip(names, restored_rgb):
        rec_gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        mse = float(np.mean((orig_gray.astype(float)-rec_gray.astype(float))**2))
        psnr = float("inf") if mse == 0 else 10*np.log10(255**2/mse)
        ssim = structural_similarity(orig_gray, rec_gray, data_range=255)
        print(f"{name}: MSE={mse:.4f}, PSNR={psnr:.2f} dB, SSIM={ssim:.4f}")
    print("Higher quantisation generally increases block artifacts and texture loss and lowers PSNR/SSIM.")

## Problem 5 — Texture-Based Segmentation using GLCM Features
Load a multi-texture image (`textures.jpg`), calculate local GLCM contrast, energy, and homogeneity, and use K-means with k=3.

In [ ]:
bgr = color_image("textures.jpg")
if bgr is not None:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    scale = min(1.0, 300/max(gray.shape))
    if scale < 1:
        size = (max(1,int(gray.shape[1]*scale)), max(1,int(gray.shape[0]*scale)))
        gray = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
        rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
    quant = (gray//16).astype(np.uint8)
    pad = np.pad(quant, 2, mode="reflect")
    h,w = quant.shape
    feats = np.zeros((h,w,3), np.float32)
    for r in range(h):
        for c in range(w):
            glcm = graycomatrix(pad[r:r+5,c:c+5], distances=[1], angles=[0],
                                levels=16, symmetric=True, normed=True)
            feats[r,c,0] = graycoprops(glcm,"contrast")[0,0]
            feats[r,c,1] = graycoprops(glcm,"energy")[0,0]
            feats[r,c,2] = graycoprops(glcm,"homogeneity")[0,0]
    X = feats.reshape(-1,3)
    X = (X-X.mean(axis=0))/(X.std(axis=0)+1e-8)
    labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X).reshape(h,w)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(rgb); plt.title("Original texture image"); plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(feats[:,:,0], cmap="viridis"); plt.title("GLCM contrast"); plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(labels, cmap="viridis"); plt.title("K-means clusters (k=3)"); plt.axis("off")
    plt.tight_layout(); plt.show()
    print("Mean local GLCM features [contrast, energy, homogeneity]:", feats.mean(axis=(0,1)))
    print("Observation: contrast measures local variation, energy reflects textural uniformity, and homogeneity is higher for similar neighbouring grey levels. Feature usefulness depends on the textures present.")

## Problem 6 — Multi-Feature Segmentation for Disease Detection
Load a plant or skin image (`plant_disease.jpg`). Combine intensity, Sobel gradient magnitude, and local 3×3 texture variance; apply K-means and display the feature maps and segmented result.

In [ ]:
bgr = color_image("plant_disease.jpg")
if bgr is not None:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    intensity = gray/255.0
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    gradient = cv2.magnitude(gx, gy)
    gradient /= (gradient.max()+1e-8)
    mean = cv2.blur(gray, (3,3))
    mean_sq = cv2.blur(gray**2, (3,3))
    variance = np.maximum(mean_sq-mean**2, 0)
    variance /= (variance.max()+1e-8)

    stack = np.stack([intensity, gradient, variance], axis=-1)
    h,w = gray.shape
    X = stack.reshape(-1,3)
    X = (X-X.mean(axis=0))/(X.std(axis=0)+1e-8)
    model = KMeans(n_clusters=3, random_state=42, n_init=10)
    segmented = model.fit_predict(X).reshape(h,w)

    plt.figure(figsize=(13,7))
    panels = [(rgb,"Original image"),(intensity,"Intensity"),(gradient,"Sobel gradient magnitude"),
              (variance,"3×3 local texture variance"),(segmented,"K-means segmentation (k=3)")]
    for i,(arr,title) in enumerate(panels,1):
        plt.subplot(2,3,i); plt.imshow(arr, cmap="gray" if arr.ndim==2 else None)
        plt.title(title); plt.axis("off")
    plt.tight_layout(); plt.show()
    print("Cluster centres in standardized feature space [intensity, gradient, texture variance]:")
    print(model.cluster_centers_)
    print("Observations: intensity separates brightness differences; gradient highlights boundaries/veins; local variance highlights irregular texture. Cluster IDs do not automatically identify healthy or diseased tissue. Similar-looking regions and lighting can cause misclassification.")

## Final checklist
- Run all six problems using the named input images.
- Save the notebook with outputs if your instructor expects evidence of execution.
- Add your own short observations based on the results.

Input files: `natural.jpg`, `smooth.png`, `city.jpg`, `textures.jpg`, and `plant_disease.jpg`.